# RoadGuardAI: Intelligent Road Safety and Pothole Detection Using AI
**BTech Capstone Project Demonstration & Academic Evaluation Notebook**

---
### Project Overview:
- **Domain**: Computer Vision, Deep Learning, Intelligent Transportation Systems
- **Core Architecture**: Ultralytics YOLO11n (You Only Look Once)
- **Hardware Acceleration**: NVIDIA GeForce RTX 4050 Laptop GPU (CUDA 12.4)
- **Dataset**: 2,083 genuine road images, 4,246 verified pothole annotations
- **Test Performance**: **73.74% mAP@50**, **71.22% Precision**, **67.54% Recall**


## 1. Environment & Hardware Verification
Check Python, PyTorch, and dedicated CUDA GPU acceleration.

In [ ]:
import sys, os
import torch
import ultralytics

print(f"Python Version:     {sys.version.split()[0]}")
print(f"PyTorch Version:    {torch.__version__}")
print(f"CUDA Available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
print(f"Ultralytics YOLO:   v{ultralytics.__version__}")

## 2. Dataset Architecture & Leakage-Free Splitting
Inspect the verified pothole dataset summary and 70/20/10 split distribution.

In [ ]:
import json
import pandas as pd

summary_path = "datasets/pothole/dataset_summary.json"
with open(summary_path, "r") as f:
    ds = json.load(f)

print("=== DATASET AUDIT SUMMARY ===")
print(f"Total Road Images:      {ds['total_images']:,}")
print(f"Total Annotations:      {ds['total_annotations']:,}")
print(f"Image Resolution:       {ds['dimension_stats']['min_width']}x{ds['dimension_stats']['min_height']}")
print(f"Unique Base Scenes:     {ds['data_quality_report']['unique_scenes']} (Leakage Prevention applied)")
print("Corrupted / Missing:    0 / 0\n")

split_df = pd.DataFrame([
    {"Split": "Train (70%)", "Images": ds['splits']['train']['images'], "Annotations": ds['splits']['train']['annotations']},
    {"Split": "Validation (20%)", "Images": ds['splits']['val']['images'], "Annotations": ds['splits']['val']['annotations']},
    {"Split": "Test (10%)", "Images": ds['splits']['test']['images'], "Annotations": ds['splits']['test']['annotations']}
])
split_df

## 3. Load Trained YOLO11n Model Checkpoint
Load the trained weights from `models/best.pt`.

In [ ]:
from ultralytics import YOLO

model_path = "models/best.pt"
model = YOLO(model_path)
print(f"[+] Model loaded successfully from: {model_path}")
print(f"Model Classes: {model.names}")

## 4. Test Evaluation & Academic Benchmarks
Genuine metrics evaluated strictly on the **unseen 10% test split (218 images)**.

In [ ]:
metrics_path = "backend/data/metrics.json"
with open(metrics_path, "r") as f:
    eval_data = json.load(f)

m = eval_data['metrics']
metrics_df = pd.DataFrame([
    {"Metric": "mAP@50 (Primary Benchmark)", "Score": f"{m['mAP50']*100:.2f}%"},
    {"Metric": "Precision (Accuracy)", "Score": f"{m['precision']*100:.2f}%"},
    {"Metric": "Recall (Detection Rate)", "Score": f"{m['recall']*100:.2f}%"},
    {"Metric": "F1-Score (Harmonic Balance)", "Score": f"{m['f1_score']*100:.2f}%"},
    {"Metric": "mAP@50-95 (Strict Localization)", "Score": f"{m['mAP50_95']*100:.2f}%"},
    {"Metric": "Unseen Test Evaluation Images", "Score": str(eval_data.get('total_test_images', 218))}
])
metrics_df

### Visualizing Evaluation Curves
Displaying the Precision-Recall (PR) Curve and Confusion Matrix generated during test evaluation.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# PR Curve
pr_path = "frontend/assets/evaluation/BoxPR_curve.png"
if os.path.exists(pr_path):
    axes[0].imshow(Image.open(pr_path))
    axes[0].set_title("Precision-Recall (PR) Curve on Test Split", fontsize=13, fontweight="bold")
    axes[0].axis("off")

# Confusion Matrix
cm_path = "frontend/assets/evaluation/confusion_matrix.png"
if os.path.exists(cm_path):
    axes[1].imshow(Image.open(cm_path))
    axes[1].set_title("Confusion Matrix (Normalized)", fontsize=13, fontweight="bold")
    axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Live Image Pothole Detection Demonstration
Run inference on a test road image and display the original vs detected potholes with bounding boxes and confidence scores.

In [ ]:
import glob
import cv2
import numpy as np
from backend.detection import detector

# Pick a sample image from the test set
test_images = sorted(glob.glob("datasets/pothole/images/test/*.jpg"))
sample_img_path = test_images[0] if test_images else "uploads/sample.jpg"

print(f"Running inference on: {sample_img_path}")
result = detector.detect_image(sample_img_path, conf_threshold=0.25)

print(f"[+] Potholes Identified: {result['pothole_count']}")
print(f"[+] Average Confidence: {result['avg_confidence']*100:.1f}%")
print(f"[+] Inference Latency:   {result['processing_time_ms']:.2f} ms")

# Display side-by-side
orig_bgr = cv2.imread(sample_img_path)
annot_bgr = result['annotated_image']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
ax1.imshow(cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB))
ax1.set_title("Original Road Image", fontsize=13)
ax1.axis("off")

ax2.imshow(cv2.cvtColor(annot_bgr, cv2.COLOR_BGR2RGB))
ax2.set_title(f"YOLO11 Detection Output ({result['pothole_count']} Potholes)", fontsize=13, fontweight="bold", color="darkred")
ax2.axis("off")

plt.tight_layout()
plt.show()

## 6. Dashcam Video Analysis Demonstration
Inspect the video processing pipeline on the sample dashcam road video.

In [ ]:
from backend.video_processing import process_video_file

sample_video = "uploads/sample_road_test.mp4"
if os.path.exists(sample_video):
    print(f"Processing video: {sample_video}...")
    v_res = process_video_file(sample_video)
    print("=== VIDEO PROCESSING RESULTS ===")
    print(f"Total Frames Processed:    {v_res['total_frames']}")
    print(f"Cumulative Potholes:       {v_res['total_potholes_detected']}")
    print(f"Max in Single Frame:       {v_res['max_potholes_in_single_frame']}")
    print(f"Average Processing FPS:    {v_res['average_fps']} FPS")
    print(f"Annotated Output:          {v_res['output_filename']}")
else:
    print("Sample video not found.")

## 7. Conclusion & Academic Evaluation Summary

1. **High Localization Precision**: Achieved **73.74% mAP@50** and **71.22% Precision** on real-world pothole imagery.
2. **Zero Data Leakage**: Enforced group-based scene isolation across 671 base scene clusters, ensuring realistic generalization.
3. **Real-time Performance**: Model processes frames in **~6.6 ms** (> 150 FPS on NVIDIA RTX 4050 GPU), making it fully viable for moving vehicles and dashcam hardware.
4. **Full-Stack Deployment**: Integrated with FastAPI backend, SQLite database persistence, and an interactive glassmorphic web dashboard.